In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cmocean
from pathlib import Path


# =========================
# 1. Load bathymetry data
# =========================
data = np.load(
r"C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\highres_bathy.npy"
)

lat2 = data[0,:,:]
lon2 = data[1,:,:]
depth2 = data[2,:,:]

# Convert to float and mask land (depth <= 0)
depth2 = np.array(depth2, dtype=float)
depth2[depth2 <= 0] = np.nan

# Used to draw coastline (land=1, ocean=-1)
elev2 = np.where(np.isnan(depth2), 1, -1)


# =========================
# 2. File paths
# =========================
npy_dir = Path(
r"C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\npy"
)

fig_dir = Path(
r"C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures"
)

fig_dir.mkdir(exist_ok=True)


# =========================
# 3. Plot domain
# =========================
lon_min = 5.2
lon_max = 5.6
lat_min = 60.75
lat_max = 60.95

levels = np.arange(0, 750, 5)


# =========================
# 4. Sill mooring location
# =========================
sill_lon = 5.2979
sill_lat = 60.8038


# =========================
# 5. Loop over CTD files and plot
# =========================
for npy_file in sorted(npy_dir.glob("*.npy")):

    CTD = np.load(npy_file, allow_pickle=True).item()

    lon_list = []
    lat_list = []
    st_list = []

    for k in CTD.keys():

        lon = float(CTD[k]["LON"])
        lat = float(CTD[k]["LAT"])
        st = int(CTD[k]["st"])

        # Keep only stations within the selected region
        if (
            lon_min <= lon <= lon_max
            and
            lat_min <= lat <= lat_max
        ):
            lon_list.append(lon)
            lat_list.append(lat)
            st_list.append(st)

    lon_arr = np.array(lon_list)
    lat_arr = np.array(lat_list)
    st_arr = np.array(st_list)

    # =========================
    # Plot
    # =========================
    fig = plt.figure(figsize=(9,8), dpi=200)

    ax = plt.axes(projection=ccrs.PlateCarree())

    ax.set_extent(
        [lon_min, lon_max, lat_min, lat_max],
        crs=ccrs.PlateCarree()
    )

    # Bathymetry
    cf = ax.contourf(
        lon2,
        lat2,
        depth2,
        levels=levels,
        cmap=cmocean.cm.deep,
        extend="max",
        transform=ccrs.PlateCarree(),
        zorder=1
    )

    # Coastline
    ax.contour(
        lon2,
        lat2,
        elev2,
        levels=[0],
        colors="black",
        linewidths=0.8,
        transform=ccrs.PlateCarree(),
        zorder=2
    )

    # Depth contours
    ax.contour(
        lon2,
        lat2,
        depth2,
        levels=[50, 100, 200, 400, 600],
        colors="gray",
        linewidths=0.5,
        alpha=0.7,
        transform=ccrs.PlateCarree(),
        zorder=2
    )

    # =========================
    # Longitude split line (5.48°E)
    # =========================
    lon_split = 5.48

    ax.plot(
        [lon_split, lon_split],
        [lat_min, lat_max],
        linestyle="--",
        linewidth=1.5,
        color="black",
        transform=ccrs.PlateCarree(),
        zorder=4
    )

    ax.text(
        lon_split + 0.003,
        lat_max - 0.01,
        "5.48°E",
        fontsize=9,
        rotation=90,
        ha="left",
        va="top",
        transform=ccrs.PlateCarree(),
        bbox=dict(facecolor="white", alpha=0.8, pad=0.2)
    )

    # Stations
    ax.scatter(
        lon_arr,
        lat_arr,
        s=38,
        color="red",
        edgecolor="black",
        linewidth=0.4,
        transform=ccrs.PlateCarree(),
        zorder=3
    )

    # Station labels
    for x, y, s in zip(lon_arr, lat_arr, st_arr):
        ax.text(
            x,
            y + 0.002,
            str(s),
            fontsize=8,
            ha="center",
            va="bottom",
            transform=ccrs.PlateCarree(),
            bbox=dict(facecolor="white", alpha=0.7, pad=0.15)
        )

    # =========================
    # Plot sill mooring
    # =========================
    ax.scatter(
        sill_lon,
        sill_lat,
        s=90,
        marker="*",
        color="yellow",
        edgecolor="black",
        linewidth=0.8,
        transform=ccrs.PlateCarree(),
        zorder=5
    )

    ax.text(
        sill_lon + 0.008,
        sill_lat + 0.003,
        "Sill mooring",
        fontsize=9,
        color="black",
        ha="left",
        va="bottom",
        transform=ccrs.PlateCarree(),
        zorder=6,
        bbox=dict(facecolor="white", alpha=0.8, pad=0.2)
    )

    # Gridlines
    gl = ax.gridlines(
        draw_labels=True,
        linewidth=0.3,
        linestyle="--",
        alpha=0.5
    )

    gl.top_labels = False
    gl.right_labels = False

    plt.title(npy_file.stem)

    cbar = plt.colorbar(cf, ax=ax, shrink=0.85, pad=0.02)
    cbar.set_label("Depth (m)")

    plt.tight_layout()

    outfile = fig_dir / f"{npy_file.stem}.png"

    plt.savefig(
        outfile,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print("saved:", outfile)

print("Done")

saved: C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures\DFN2021460.png
saved: C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures\GOS2012115.png
saved: C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures\GOS2014117.png
saved: C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures\GOS2017114.png
saved: C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures\GOS2018111.png
saved: C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures\GOS2019114.png
saved: C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures\GOS2020112.png
saved: C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures\GOS2022112.png
saved: C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures\GOS2023001013.png
saved: C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures\GOS2024001014.png
saved: C:\Users\quzho2904\OneDrive